In [1]:
# librerias
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.io as pio
from IPython.display import Image

# spark 
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("stock").getOrCreate()

In [2]:
# serie de precios de nvidia del ultimo año
nvda = yf.download('NVDA', period='1y')
nvda

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA
Date,,,,,
2024-02-22,78.515747,78.552732,74.198970,75.003742,865100000
2024-02-23,78.794655,82.370637,77.548007,80.767095,829388000
2024-02-26,79.069588,80.623148,78.482748,79.677409,503973000
2024-02-27,78.678688,79.457474,77.140130,79.358496,391705000
2024-02-28,77.640991,78.910628,77.103142,77.598004,393110000
...,...,...,...,...,...
2025-02-14,138.850006,139.250000,135.500000,136.479996,195479600
2025-02-18,139.399994,143.440002,137.929993,141.270004,219176600


In [5]:
# el df tiene un multiindex, por lo que es necesario ajustar el head del df
nvda.columns = nvda.columns.to_flat_index() # convertir el multiindex en una sola columna
nvda = nvda.rename(columns={('Close', 'NVDA'): 'Close', 
                            ('High', 'NVDA'): 'High', 
                            ('Low', 'NVDA'): 'Low', 
                            ('Open', 'NVDA'): 'Open', 
                            ('Volume', 'NVDA'): 'Volume'}
) # renombrar las columnas

In [6]:
# se resetea el indice de las columnas para convertir la fecha en una columna
nvda = nvda.reset_index()

In [ ]:
# crear un df de spark
df_spark_nvda = spark.createDataFrame(nvda)
display(df_spark_nvda)

DataFrame[Date: timestamp, Close: double, High: double, Low: double, Open: double, Volume: bigint]

In [ ]:


# Crear la gráfica con Plotly
fig = px.area(nvda, x='Date', y='Close', title='Precio de Cierre de NVIDIA - Último Año')

# Personalizar la gráfica
fig.update_layout(xaxis_title='Fecha', yaxis_title='Precio de Cierre (USD)')
fig.update_traces(line=dict(color='#C78590'))  # Color pastel oscuro (rosa)

# **Guardar la gráfica como imagen estática**
fig.write_image("grafica_nvda.png")  # Guarda la imagen en el directorio actual

# **Mostrar la imagen dentro del Jupyter Notebook**
display(Image("grafica_nvda.png"))